# 第13回: 準線と離心率 — 変わる二つの距離、変わらない一つの比（隠れた直線が立体から現れる）

**主題**: L12 で、天下りだった二焦点が Dandelin 球の **接点** として立体から現れた。だがもう一人の隠れた登場人物が残っている —— **準線（directrix）** である。L7 で「隠れた直線」として予告し、L11 で SymPy により $x = \pm a^2/c$ と式の上で一点だけ確かめた、あの直線だ。今日は準線が、Dandelin 球が円錐に接する **接触円を含む平面** と切断面の **交わり** として立体から現れることを見る（Dandelin–Quetelet）。そして楕円上の点 $P$ をどこへドラッグしても、$|PF_1|$（焦点までの距離）も準線までの距離も **変わる** のに、その **比** だけは **変わらない** —— この一つの数（離心率 $e$）が、楕円・放物線・双曲線の三曲線を一列に並べる。

**副題**: 今日の道具立ては「**取り出す → 動かす → 比で言い切る**」。新しいソフトウェア機能は足さない（L11 までの ggblab・SymPy で足りる）。足すのは一つの視点 —— **変わる量の中に、変わらない量を探す**。距離そのものは動く。比が動かない。この「何が動いて何が動かないか」を手で確かめる作法は、作図に限らず、式・データ・プログラムのどれを調べるときにも効く（→ L14 でさらに研ぐ）。

**学習目標**

1. L12 の接触円 $k_1$ を含む水平面と切断面の **交わりの直線** として、準線を立体から取り出せるようになる。
2. 楕円上の点 $P$ をドラッグし、$|PF_1|$ と $d(P, \text{準線})$ が **両方変わる** のに比 $|PF_1| / d(P, \text{準線})$ が **一定** $= e \approx 0.3954$（L11/L12 の $c/a$ と一致）であることを確かめられるようになる。
3. 上の球（接触円 $k_2$）からも **第二の準線** が出て、$F_2$ との組で同じ比 $e$ を与えることを確かめられるようになる。
4. 切断面の傾きを変えると $e$ が $e<1$（楕円）→ $e=1$（放物線・切断面が母線と平行）→ $e>1$（双曲線）と **連続に** 動き、一つの数が三曲線を貫くことを、予測してから確かめられるようになる。

```{admonition} 目標の確認 — @Codex で達成目標を引く
:class: note
本回の達成目標を **@Codex に lancedb-rag（教材ドメイン RAG）で確認**してもらえる（L5 から運用）。
（プロンプト例）@Codex この回（L13 準線と離心率）の達成目標を lancedb-rag で調べて、要点を整理して。
特に「準線 = 接触円の平面と切断面の交わり」「距離は変わるが比は変わらない」「e が三曲線を貫く」が
腑に落ちるか、自分の理解と照らしてから先へ進む。
```

## 0. 前回 (第 12 回) の振り返り — 焦点は出た。準線が残っている

前回、二焦点は Dandelin 球と切断面の **接点** として立体から現れた。「距離の和が一定」は母線に沿った接点間の長さ $|AB|$ だった。そして枕で、下の球が円錐に接する **接触円 $k_1$**（高さ $z \approx 1.611$ の水平な円）だけを予告して終えた。

L10/L11 にはもう一つ、宙に浮いたままの登場人物がいる。**準線** —— L11 で「焦点までの距離と準線までの距離の比が一定 $=e$」を SymPy で一点だけ確かめ、$x = \pm a^2/c$ という式も出た。だが、その直線が **立体のどこから来るのか** は、まだ誰も見ていない。

> **今日の問い**: 準線は、円錐の三次元構成の **どこ** にあるのか。そして「比が一定」は、どの範囲まで成り立つのか。

答えの入口は、枕に置いた接触円 $k_1$ である。焦点が「球と切断面の接点」から来たように、準線は「球と円錐の **接触円を含む平面**」から来る。

## 1. ggblab セットアップと L12 の構成の再現

In [ ]:
using Pkg
Pkg.activate("../..")   # この教材プロジェクトの環境を有効化
Pkg.resolve()
Pkg.instantiate()       # 必要なパッケージを用意（初回は少し時間がかかる）
using GeoGebra
ENV["GGB_DIRECT_TRANSPORT"] = "true"   # ggblab とアプレットの直接通信を有効化

In [ ]:
inject_applet()

```{admonition} 3D ビューに切り替える（手作業・L12 と同じ）
:class: warning
本回も **3D の作図** です。アプレット左上の **メニュー（☰）→「アプリの切り替え」→「空間図形」** を選び、
**空間図形（3D ビュー）** に切り替えてから先へ進んでください。
```

In [ ]:
# L12 と同じ構成（円錐・切断面・二球・二焦点・接触円 k1）を一気に再現する。
# 数値は L12 著者検証済みの確定値。規律1: 各オブジェクトを一意のシンボルに束縛。
@ggb :const :new
@ggb apex=Point("{0, 0, 0}")
@ggb top=Point("{0, 0, 5.4}")
@ggb cone=Cone(:top, :apex, 1.9712)        # 半開角 0.35 rad
@ggb nrm=Vector((0.4, 0, 1))
@ggb pl=PerpendicularPlane((0,0,2.5), :nrm)
@ggb section=IntersectConic(:pl, :cone)    # 切り口の楕円
@ggb C1=Point("{0, 0, 1.8257}")
@ggb sph1=Sphere(:C1, 0.6260)              # 下の Dandelin 球
@ggb C2=Point("{0, 0, 3.9639}")
@ggb sph2=Sphere(:C2, 1.3592)              # 上の Dandelin 球
@ggb perp1=PerpendicularLine(:C1, :pl)
@ggb F1=Intersect(:perp1, :pl)             # 焦点その1（下球の接点）
@ggb perp2=PerpendicularLine(:C2, :pl)
@ggb F2=Intersect(:perp2, :pl)             # 焦点その2（上球の接点）
@ggb nrm0=Vector((0, 0, 1))
@ggb Ck1=Point("{0, 0, 1.6111}")
@ggb k1=Circle(:Ck1, 0.5881, :nrm0)        # 下球の接触円（L12 の枕）

**観察1**: L12 の景色が戻ってきた。今日の主役は、最後に描いた水平な円 `k1` —— 下の球が円錐の内壁にぐるりと接している円である。

## 2. 準線を立体から取り出す —— 接触円の平面と切断面の交わり

### 2.1 交わりの直線

接触円 `k1` は高さ $z \approx 1.611$ の **水平面** の上に載っている。その水平面と、斜めの切断面 `pl` は、どちらも平面だから、交わりは **一本の直線** になる。それを取る。

In [ ]:
# 接触円 k1 を含む水平面と、切断面 pl の交わりの直線を取る。これが準線（になるはず）。
@ggb pk1=PerpendicularPlane(:Ck1, :nrm0)   # k1 を含む水平面（中心を通り、法線が軸方向）
@ggb dir1=Intersect(:pk1, :pl)             # 水平面 ∩ 切断面 = 一本の直線

**観察2**: 直線 `dir1` は切断面の上に現れるが、楕円 `section` の **外側**（$x \approx 2.22$ のあたり）を通る。楕円には触れない。焦点が楕円の **内側** の点だったのに対し、この直線は **外側** にいる —— 焦点と準線は、楕円を挟んで内と外に立つ。

### 2.2 動かす —— 距離は変わる、比は変わらない

L11 で式の上だけ確かめた「比一定」を、いま立体の上で動かして確かめる。楕円上の点 $P$ から、焦点 $F_1$ までの距離と、直線 `dir1` までの距離を測り、その比を見る。

```{admonition} 予測を書き留める —— 動かす前に
:class: important
$P$ をドラッグすると、$|PF_1|$ は変わる（焦点に近づいたり遠ざかったりする）。準線までの距離も変わる。
では **比** はどうなるか —— 変わるか、変わらないか。変わらないとしたら値はいくつか。
**セルを実行して動かす前に**、自分の予測を一行書き留めてから進む（L12 までに拾った値が手掛かりになる）。
```

In [ ]:
# P をドラッグして、二つの距離と、その比を見る。
@ggb P=Point(:section)                     # 切り口の楕円上の点（ドラッグできる）
@ggb dPF=Distance(:P, :F1)                 # 焦点までの距離（変わる）
@ggb dPd=Distance(:P, :dir1)               # 準線までの距離（変わる）
@ggb ratio="dPF / dPd"                       # 比（さて、どうなるか）

**観察3**: $P$ をどこへ動かしても `ratio` は $\approx 0.3954$ から動かない。分子も分母も動いているのに、比だけが止まっている。そしてこの $0.3954$ は、L11 で方程式から計算した **離心率 $e = c/a$**、L12 の作図値 $c/a = 0.3971/1.0043$ と一致する。**L7 で「隠れた直線」と呼び、L11 で式の上に現れた準線が、いま立体の中の実在の直線として、楕円の外側に立っている**。

### 2.3 数値でも確かめる —— どの位置でも同じ比

In [ ]:
# §2.2 の「比一定」を Julia でも数値で確かめる（カーネルは Julia）。
# 楕円上の点を母線の角度 φ でパラメータ化し、|PF1| と d(P, 準線) と比を並べる。
nx, d, α = 0.4, 2.5, 0.35
F1 = (0.23251, 0.0, 2.40700)               # L12 の焦点（下球の接点）
xd, zd = 2.22233, 1.61107                  # 準線: 直線 {x=xd, z=zd}（y 軸に平行）
for φ in (0.0, 0.7, 1.4, 2.1, 2.8)
    z = d / (1 + nx * tan(α) * cos(φ))     # 切り口上の点（L11 と同じパラメータ化）
    P = (z * tan(α) * cos(φ), z * tan(α) * sin(φ), z)
    dPF = sqrt(sum((P .- F1).^2))
    dPd = sqrt((P[1] - xd)^2 + (P[3] - zd)^2)   # 準線は y 軸平行 → (x,z) 面で測る
    println("φ = $φ:  |PF1| = ", round(dPF, digits=5),
            "  d(P,準線) = ", round(dPd, digits=5), "  比 = ", round(dPF/dPd, digits=6))
end

**観察4**: 距離の列は二つとも大きく動くのに、比の列は $0.395360$ で微動だにしない。L10 の「和が一定」（$|PF_1|+|PF_2| = 2a$）に続いて、二つ目の「一定」が手に入った —— **比が一定** $= e$。和の一定は二焦点の言葉、比の一定は焦点一つ + 準線一本の言葉である。同じ楕円が、また別の言葉で言い切れた。

### 2.4 なぜ比が一定か —— L12 と同じ一手がもう一度効く

理由は L12 §3.4 と **同じ一手** から出る。切り口の点 $P$ から下の球への接線として $|PF_1| = |PA|$（$A$ は $P$ を通る母線が接触円 $k_1$ を貫く点）。つまり焦点までの距離は、**母線に沿って接触円の平面まで降りる長さ** に等しい。一方、準線までの距離は、**切断面の中で** 接触円の平面（の交わりの直線）まで降りる長さである。どちらも「$P$ から同じ水平面（高さ $z_1$）までの距離」を、違う傾きの方向で測っている —— 母線に沿って測るか、切断面に沿って測るか。二つの傾きは $P$ によらず一定（円錐の半開角と、切断面の傾き）だから、**比も $P$ によらず一定** になる。これが Dandelin–Quetelet の見立てである。

$$\frac{|PF_1|}{d(P,\text{準線})} = \frac{|PA|}{d(P,\text{準線})} = \frac{\text{母線に沿った降り方}}{\text{切断面に沿った降り方}} = e \quad (\text{一定})$$

## 3. 第二の準線 —— 上の球からも同じものが出る

焦点が二つあったように、準線も二本ある。上の球の接触円 $k_2$（高さ $z \approx 3.498$）を含む水平面と切断面の交わりが、第二の準線である。こちらは $F_2$ と組む。

In [ ]:
# 上球の接触円 k2 とその平面、第二の準線。F2 との比も同じ e になるか。
@ggb Ck2=Point("{0, 0, 3.4979}")
@ggb k2=Circle(:Ck2, 1.2768, :nrm0)        # 上球の接触円
@ggb pk2=PerpendicularPlane(:Ck2, :nrm0)
@ggb dir2=Intersect(:pk2, :pl)             # 第二の準線（楕円の反対側の外）
@ggb dPF2=Distance(:P, :F2)
@ggb dPd2=Distance(:P, :dir2)
@ggb ratio2="dPF2 / dPd2"                    # こちらも 0.3954 で一定になるか

**観察5**: `ratio2` も $\approx 0.3954$ で一定。第二の準線は楕円の **反対側** の外（$x \approx -2.49$）に立ち、$F_2$ と組んで同じ比を与える。焦点と準線は「内側の点と外側の直線」の **組** で現れ、その組が楕円を挟んで左右に一つずつある —— L12 §5 で見た「図全体の鏡映対称」が、ここでも効いている。

## 4. 一つの数が三つの曲線を貫く —— 傾きを変える

### 4.1 予測 —— 切断面を母線と平行まで傾けたら

L12 の思考課題4で「準線が Dandelin 球からどう出てきそうか」の見立てを残した。今日それが出た。次はもう一歩 —— 切断面の **傾き** を変える。いまの傾きは `nrm=(0.4, 0, 1)` だった。この $0.4$ を大きくしていくと、切断面はどんどん立ち上がり、あるところで **母線と平行** になる。

```{admonition} 予測を書き留める —— 傾ける前に
:class: important
傾きを $0.4$ から増やしていくと、比 $e$ はどうなるか。切断面が **母線とちょうど平行** になった瞬間、
切り口はどんな曲線になるか。さらに傾けて母線を **越えたら** 何が起こるか。
L11 で三曲線の離心率を式から計算したこと、L12 §6 の「双曲線は反対側の円錐も切る」を手掛かりに、
**三段階の予測**（平行の手前 / ちょうど平行 / 越えた後）を書き留めてから、次のセルへ進む。
```

### 4.2 確認 —— e の値の三段変化

3D 作図で切断面を作り直してもよいが、ここでは比の値だけを数値で追う（作図での確認は課題に回す）。切断面が母線と平行になる傾きは $n_x = \sqrt{1/\sin^2\alpha - 1} \approx 2.7395$ である（思考課題2で導出）。

In [ ]:
# 傾き nx を変えながら、切り口の上の点で 比 |PF1|/d(P,準線) を計算する。
# 準線・焦点の位置は傾きごとに決まる（下球一個から出す。式は著者ノート参照）。
α = 0.35
function ecc_of(nx; d=2.5, φ=0.5)
    nn = sqrt(nx^2 + 1)
    h1 = d / (1 + nn * sin(α))             # 下球の中心の高さ（傾いても分母 > 1 で安全）
    tt = (h1 - d) / (nx^2 + 1)
    F1 = (-tt * nx, 0.0, h1 - tt)          # 焦点 = 球と切断面の接点（L12 §3.2 の垂線の足）
    z1 = h1 * cos(α)^2                     # 接触円の高さ
    xd = (d - z1) / nx                     # 準線の x（水平面 z=z1 と切断面の交わり）
    z = d / (1 + nx * tan(α) * cos(φ))
    P = (z * tan(α) * cos(φ), z * tan(α) * sin(φ), z)
    dPF = sqrt(sum((P .- F1).^2))
    dPd = sqrt((P[1] - xd)^2 + (P[3] - z1)^2)
    return dPF / dPd
end
nx_parallel = sqrt(1/sin(α)^2 - 1)         # 母線と平行になる傾き ≈ 2.7395
for nx in (0.4, 1.0, 2.0, nx_parallel, 3.5614)
    println("nx = ", round(nx, digits=4), ":  e = ", round(ecc_of(nx), digits=6))
end

**観察6**: $e$ は傾きとともに **連続に** 増えていき、母線と平行になる傾き $n_x \approx 2.7395$ で **ちょうど $1$** になり、越えると $1$ を **超える**。$e<1$ が楕円、$e=1$ が放物線、$e>1$ が双曲線 —— L11 で三つの方程式から別々に計算した離心率が、ここでは **一つの構成の中の一つのつまみ**（切断面の傾き）を回すだけで、連続に移り変わる。三つの曲線は三つの別物ではなく、**一つの数 $e$ の三つの区間** だった。境界（$e=1$・放物線）は特別な曲線というより、楕円と双曲線の **あいだの一瞬** である。

```{important} 定義・命題のまとめ（L13 の到達点）
- **準線** ＝ Dandelin 球の **接触円を含む平面** と切断面の **交わりの直線**（焦点＝接点、準線＝接触円の平面。どちらも球から出る）。
- **比一定**: 切り口の任意の点 $P$ で $|PF| / d(P, \text{準線}) = e$（焦点・準線は同じ側の球の組で使う）。理由は L12 と同じ「接線の等長」＋ 二つの傾きの比。
- **離心率 $e$ が三曲線を貫く**: 切断面の傾きを上げると $e$ は連続に増え、$e<1$ 楕円 / $e=1$ 放物線（母線と平行）/ $e>1$ 双曲線。
- 準線は楕円の **外** に立つ（焦点は内、準線は外。この「外にある」ことが §5 の歴史に効く）。
```

## 5. 停滞と画期の照射 —— 焦点概念の四つの層と、円錐の「外」に立つ準線

### 5.1 「焦点には何があるか」—— 四つの層

L12 の思考課題5 で「楕円の焦点には何があるか（片方は太陽だが）」と問うた。実は「焦点」という概念そのものが、二千年かけて **四つの層** で積み上がってきたものである。そしてこの学期、皆さんはその四層を（歴史と同じ順ではないが）全部歩いている:

| 層 | 誰が・いつ | 何をしたか | この学期のどこで |
|---|---|---|---|
| 第 I 層 | Apollonius（前 3 世紀末） | 距離の **和（差）一定** という **性質** を証明（「適用から生じる点」と呼んだ。作図法ではなく性質として） | L10 の定義・L11 の式 |
| 第 II 層 | Pappus（4 世紀） | **焦点・準線性質** —— 焦点距離と準線距離の比が一定（離心率）—— を初めて記述 | **今日 L13** |
| 第 III 層 | イスラム期（al-Sijzī, Ibn Sinān・9 世紀） | **紐による楕円作図**の最初の明示的記述 —— 性質から **作図法** への概念的飛躍 | L10 の紐 |
| 第 IV 層 | Kepler（1609） | **focus**（炉・火床）という **物理的命名** と、惑星軌道で太陽がこの点にあるという **天文学的同定** | L10 の「Kepler の発見」 |

注意してほしいのは、Kepler の貢献が「点の発見」ではないことだ。Kepler が「太陽は中心ではなく **別の特別な点** にある」と気づくためには、楕円に中心とは別の特別な点があることを **既に知っていなければならなかった** —— 彼はそれを Apollonius の『Conica』第 III 巻から知っていた。第 I 層がなければ第 IV 層はない。各層はバラバラの発見ではなく、**前の層の上にしか積めない**。そして今日皆さんがやったこと（準線を立体から取り出し、比一定を確かめる）は、第 II 層 —— この四層の中で一番影の薄い、Pappus の層 —— を、Dandelin–Quetelet（1822）の球で **立体に接地させ直す** ことだった。Pappus は Apollonius と Kepler のあいだの長い時間を埋める **中継点** なのに、しばしば埋もれる。今日、少なくともこの教室では埋もれなかった。

### 5.2 準線は、円錐の「外」にあった

第 II 層には不思議がある。焦点-準線性質は Pappus が伝える Euclid の失われた著作『Surface Loci』への注記に既に現れる —— つまり **性質そのものは Apollonius の前後から知られていた**。それなのに、Apollonius の大著『Conica』（全 8 巻・487 命題）には、**準線がほとんど現れない**。切り口・軸・通径・接線・漸近線を網羅した百科事典的大著が、現代の教科書なら最初に教える性質を素通りしている。なぜか。

一つの **診断**（証明された因果ではなく、読みである —— この区別は L14 と最終レポートで効いてくる）はこうだ: **準線は円錐の「外」にあるから**。Apollonius の方法は、円錐の **中** の構造（切り口と軸と通径）から曲線の性質を汲み上げる。焦点はまだ切り口の内側の点だが、準線は切り口の外、いわば **円錐の外の虚空** に立つ直線である。円錐の中を見る言葉で組み立てられた体系には、外に立つ直線の居場所がなかった。今日われわれが見たように、準線を立体に **接地** させるには「球の接触円の平面」という補助対象が要る —— それが見つかるのは、二千年あまり後の 1822 年だった。

これは L12 §6 の「二千年の見落とし」の続きであり、もう一つの教訓でもある。**体系がどれほど網羅的でも、その体系の「見る場所」の外にあるものは扱えない**。Apollonius の 487 命題は円錐の中を見尽くしたが、準線は外にいた。—— では、いまわれわれが使っている道具や体系は、どこを「見て」いて、何がその外に立っているだろうか。この問いは最終回（L15）の発表と議論で戻ってくる。

```{admonition} 進捗の確認 — セル出力を @Codex に読ませる
:class: tip
@Codex は **jupyter-server-mcp であなたのセル入出力を直接読みます**（プロンプトだけではない）。
（プロンプト例）@Codex ここまでのセル出力を読んで、達成目標①（準線を交わりの直線として取り出した）
②（距離は変わるが比は一定 ≈ 0.3954）③（第二の準線でも同じ比）④（傾きで e が 1 を連続に跨ぐ）に
どこまで到達したか、まだ埋まっていないセルはどこか、具体的に挙げて。
```

```{admonition} 今回の課題
:class: tip

**必修**
1. 接触円 $k_1$ を含む水平面と切断面の交わりとして準線 `dir1` を取り出し（§2.1）、$P$ をドラッグして比 $|PF_1|/d(P,\text{準線})$ が一定 $\approx 0.3954$ であること、その値が L11/L12 の $c/a$ と一致することを確かめよ（§2.2–2.3）。
2. 第二の準線 `dir2` を上球から取り出し、$F_2$ との組で同じ比になることを確かめよ（§3）。さらに §4.2 のセルで、傾き $n_x$ を動かすと $e$ が $1$ を連続に跨ぐことを確かめよ。

**思考課題**
3. §2.4 の「二つの傾きの比」の議論を、自分の言葉で書き直せ。特に「母線に沿った降り方」と「切断面に沿った降り方」が、それぞれ **どの角度** で決まるかを図で示せ（ヒント: 半開角 $\alpha$ と、切断面の傾き）。
4. 切断面が母線と平行になる傾きが $n_x = \sqrt{1/\sin^2\alpha - 1}$ であることを導け（ヒント: 切断面の法線 $(n_x, 0, 1)$ と母線の方向ベクトルが直交する条件）。
5. **(発見してほしい問い)** $e \to 1$（放物線）のとき、**上の球と第二の準線はどこへ行くか**。§4.2 の式で $n_x$ を $2.7395$ に近づけながら、上球の中心の高さ $h_2 = d/(1 - \sqrt{n_x^2+1}\,\sin\alpha)$ の分母に何が起こるかを観察し、「放物線に焦点が一つしかない」ことと結びつけて、自分の言葉で書き留めよ（L14–L15 の議論の素材）。
```

```{admonition} 学期末レポート出題 —— 科学史転生無双・最終版（L15 で発表）
:class: important

L4 §7 で始めた **科学転生無双レポート** を、学期末レポートとして改めて出題する。今回は転生先を自分で選ぶ。

**課題**: この学期に歩いた系譜から **一人** を選んで転生し、**その人が半歩先で見られなかったもの** を、その人の時代の道具 + 皆さんの現代知識 + `@Codex` 共創で見せる文章（A4 数枚）を書け。

- **転生先の候補**（§5.1 の四層と、これまでの回の登場人物から）: Menaechmus / Euclid / Archimedes / Apollonius / **Pappus** / al-Sijzī・Khayyam（イスラム期） / Kepler / Descartes / Fermat / Newton / Dandelin / Hamilton / Feynman。
- **「半歩先」の例**: Apollonius に準線を見せる（今日の接触円の平面を、彼の言葉でどう説明する?）。Euclid に「接するとき交点は二つ重なっている」を見せる（→ 次回 L14 が丸ごと素材になる）。Kepler に、彼が手計算でやった系統的パターン抽出を現代の道具で再演して見せる。Newton に速度の円を見せる（→ L15）。Pappus に「あなたの層は二千年後に主役になる」と伝える。
- **規律は L4 §7 と同じ**: `@Codex` と協力して構成、ただし最後の articulation（決め台詞）は自分の語彙で。末尾に AI とのやり取り要点と「何を AI に任せ、何を自分で考えたか」を明示。
- **無双可能性の自己評価**: 書き始める前に、「この転生先で本当に無双できるか」を `@Codex` 自身にも自己評価させよ（L4 §7.3 で見た、AI が Turing では無双できないと正直に認めて Neumann に乗り換えた逸話を思い出すこと）。AI が無双できる場面とできない場面の **境界** が、そのままレポートの厚みになる。
- **評価軸**: L4 §7.2 の三軸（伝達可能性 / synthesis の独自性 / AI 役割分担の自覚）に、今回は一つ足す —— **反証可能性**: 「この人はこれを見られなかった」という主張の根拠と、自分の再構成が **間違っていたら何で分かるか**。
- **日程**: 次回 L14 で **冷間チェックの作法**（書いた直後の熱を冷ましてから自分の一番痛い穴を探す方法）を渡す。**L15 で発表**（形式は L15 で告知）。提出詳細は LMS。
```

:::{important} 授業末尾の自己評価 —— `@Codex` に聞いてみる（任意、L4–L12 から継続）

L4–L12 と同じ template で、本回の自己評価を試してください。**任意**です。

````text
@Codex 今日のノートブックを全 cell 読んで評価してください。
次の三つを区別して articulate してください:

1. 自分で考えて書いた cell — 思考の痕跡が残っている部分
2. AI 委託で書いたが、理解して受け入れた cell — 動いて、なぜ動くか説明できる部分
3. AI 委託で書いたが、なぜ動くか説明できない cell — 動いているが、理解で未到達の部分

加えて: 今日の主題（準線 = 接触円の平面と切断面の交わり / 距離は変わるが比は変わらない /
第二の準線と F2 の組 / 傾きで e が 1 を連続に跨ぐ）に対する到達度、
完成しないまま残った問い、次回（L14 接線と判別式）への接続点。

特に「動かす前に書いた予測」と「動かした後の観察」がどこで一致し、どこでずれたかを
誤魔化さず正直に articulate してください。
最終判断は自分で。@Codex の照合は候補の提示であって、評価は自分の構成(construction)に対して行う。
````

JupyterAI のやり取り log は LMS 経由で先生に届きます —— 学期末レポート・発表（第 15 回）の materials として毎週蓄積。
:::

## 6. 次回への接続

今日、隠れていた最後の登場人物（準線）が立体から現れ、離心率 $e$ という一つの数が三曲線を貫いた。L10 からの円錐曲線の登場人物は、これで全員が立体に接地した —— 焦点（接点）、和一定（母線長）、準線（接触円の平面）、離心率（二つの傾きの比）。

だが、まだ一度も正面から問うていない言葉が残っている。L4 から何度も使い、L12 の証明の芯でもあった、あの言葉 —— **「接する」** とは、そもそも何か。円と直線が「接する」とき、交点は **何個** あるのか。一個? では、交わる場合の二個から、離れる場合の零個へ、どうやって「一個」を通り抜けるのか。

次回（第 14 回）は、この素朴な問いに GeoGebra が **奇妙な答え** を返すところから始める。そしてその奇妙さが、Euclid 以来二千年ものあいだ誰も言葉にできなかったものの正体であり、高校で習ったある式（判別式）がその二千年を締めくくる到達点だったことを見る。あわせて、学期末レポート（科学史転生無双・最終版）の **冷間チェックの作法** を渡す —— 五年かけて紡がれた物語でさえ、冷えてから見直すと太字の一行に穴が見つかることがある、という実話つきで。

## 枕（次回への予告）—— 円の外の点から、接線を引く

次回の主役は、L4 以来の旧友 —— 円の接線である。外の点から円へ接線を引く作図（Thales の円を使う、あの手つき）だけ、2D の別画面で予告しておく。

In [ ]:
# 次回の枕: 円の外の点 P から円 c への接線(2D)。Thales の円との交点が接点になる。
# （なぜ Thales でうまくいくのか — L4 の「半円に内接する角は直角」を思い出しておく）
@ggb :const :new
@ggb O=(0, 0)
@ggb c=Circle(:O, 2)                       # 半径 2 の円
@ggb P=(4, 0)                              # 円の外の点（次回ドラッグする）
@ggb M=Midpoint(:O, :P)
@ggb th=Circle(:M, :O)                     # Thales の円（OP を直径とする円）
@ggb l_t="{Intersect(c, th)}"
@ggb T_1=l_t(1)
@ggb T_2=l_t(2)                            # 二つの交点 = 二つの接点（になるはず）
@ggb t_1=Line(:P, :T_1)                    # 接線その1
@ggb t_2=Line(:P, :T_2)                    # 接線その2

**読み方**: 直線 `t_1`, `t_2` は円 `c` に「触れているだけ」に見える。次回はまず、この「触れているだけ」を GeoGebra 自身に数えさせる —— 円と接線の **交点** を求めさせると、何が返ってくるか。そこから二千年の物語が始まる。

## 参考文献

- [Dandelin spheres - Wikipedia](https://en.wikipedia.org/wiki/Dandelin_spheres)（接触円の平面と切断面の交わりとしての準線。Dandelin–Quetelet）
- [Directrix (conic section) / Conic section - Wikipedia](https://en.wikipedia.org/wiki/Conic_section#Eccentricity,_focus_and_directrix)（焦点-準線性質と離心率。$e$ による三曲線の統一）
- [Adolphe Quetelet - Wikipedia](https://en.wikipedia.org/wiki/Adolphe_Quetelet)（準線まで含めた形は Dandelin–Quetelet の定理。統計学者になる前の幾何学者としての仕事）
- [Pappus of Alexandria - Wikipedia](https://en.wikipedia.org/wiki/Pappus_of_Alexandria)（Euclid『Surface Loci』への注記に焦点-準線性質。古代に知られていた形跡）
- [Apollonius of Perga - Wikipedia](https://en.wikipedia.org/wiki/Apollonius_of_Perga)（『Conica』487 命題に準線がほぼ現れないこと）
- 前回 第12回（Dandelin 球・接点 = 二焦点）は @Codex に lancedb-rag で「第12回 Dandelin 球 接点 二焦点 母線 和一定」を尋ねる（project=textbook）
- 第11回（SymPy・方程式・準線の式 $x = \pm a^2/c$）は @Codex に lancedb-rag で「第11回 円錐 切断 SymPy 準線 離心率」を尋ねる（project=textbook）
- 準線が「円錐の外」にあり古代の体系から漏れたことの背景は @Codex に lancedb-rag で「準線 円錐の外 虚空 corpus gap 2200年」を尋ねる（project=conversations）
- §5.1 の四層構造（Apollonius／Pappus／イスラム期／Kepler）の原典は @Codex に lancedb-rag で「焦点概念の重層性 四層 Pappus 中継点 数学史対話」を尋ねる（project=conversations） —— 先生の五年の物語紡ぎの総括（2026-05-11 discussion_memo §6）と、その冷間再検証（2026-07-03 REVIEW）
- 教材オーサリング規約は @Codex に lancedb-rag で「教材オーサリング規約 ggblab セル シンボル 規律1」を尋ねる（project=textbook）

```{admonition} 著者ノート —— 数値は判別器パッケージで外部検証済み（学生用ではない）
:class: note

本回の数値（焦点・準線・離心率・傾き掃引）は、GeoGebra とは独立に `antidescartes` パッケージ
（`scene_dag.build_conic_fd_graph` / `lower_focus_directrix`・研究用の依存 DAG 実装）で検証してから確定した:

- 焦点 $F_1 = (0.23251, 0, 2.40700)$、接触円高さ $z_1 = 1.61107$（L12 の $1.6111$ と一致）、準線 $x = 2.22233$。
- 上球側: $z_2 = 3.49785$、第二準線 $x = -2.49463$、接触円 $k_2$ 半径 $= z_2\tan\alpha = 1.2768$（$k_1$ 側は $z_1\tan\alpha = 0.58809$ で L12 の $0.5881$ と一致）。
- 比 $e = 0.395360$ が **両焦点・両準線・全ドラッグ位置（φ 掃引）で一定**（広がり $<10^{-9}$）。L12 の $c/a = 0.39540$ と一致。
- 傾き掃引: $n_x = 0.4 \to e = 0.395$、$n_x = 2.7395$（母線平行）$\to e = 1.000000$、$n_x = 3.5614 \to e = 1.0249$。
  遷移は解析境界 $n_x = \sqrt{1/\sin^2\alpha - 1}$ に正確に載る（`tests/test_trace_discriminator.py` の遷移位置テストと同一の検証）。

**本回の教材構成そのものが antidescartes の適用第一号である（研究側 register・本文には出さない）**:
§2.2 の「予測を書き留めてから動かす」= predict-then-reveal の平語化。§2.2–2.3 の「距離は変わるが比は
変わらない」= drag-invariance oracle（ecc は drag 変数 φ の不変量・v3-A5 の A5 register）の平語化。
§4 の「e が 1 を連続に跨ぐ」= 境界を貫く継続（continuous_through_boundary）の平語化。内部用語
（Construct/Sample/oracle/register 等）は学生向け本文に一切出していない（平語 lint 済み・G-C 授業投入は先生 review 後）。

**discussion_memo の埋め込み（2026-07-06 改版）**: §5.1 の四層表 = discussion_memo §6（焦点概念の重層性・
先生の最終的反省点「Kepler は点の発見でなく命名と同定」を含む）の教材化。Pappus 中継点の再評価も同 §6。
§5.2 の「診断（証明された因果ではなく、読み）」の register は冷間スイープ F1（封印テーゼは L3・因果事実の
register で書かない）の反映。学期末レポート出題（科学史転生無双・最終版）は先生指示（2026-07-06）—— L4 §7 の
inaugural を最終版として再出題、L14 冷間チェック → L15 発表の三段。転生先候補と「半歩先」例は memo §9 年表から。

**実機確認が必要な点（授業前・G-C gate）**: (1) `Intersect(平面, 平面)` が 3D ビューで交線を返すこと
（L12 の `IntersectConic` 引数順の轍。もし不可なら `IntersectPath(pk1, pl)` を試す）。(2) `Distance(点, 直線)`
の 3D 挙動。(3) 枕の 2D セルは「アプリの切り替え」で平面ビューに戻す必要があるか。
```